# 2.1 The Unicode Standard

## 2.1 a) What Unicode character does chr(0) return

### Unicode character chr(0) returns =  . It is a null byte


In [12]:
chr(0)

'\x00'

In [13]:
print(chr(0))

 


In [8]:
"this is a test" + chr(0)

'this is a test\x00'

In [5]:
print(f"Unicode character chr(0) returns = {chr(0)} . It is a blank")

Unicode character chr(0) returns =   . It is a blank


In [9]:
print("this is a test"+chr(0))

this is a test 


## 2.1 b) How does this character’s string representation (__repr__()) differ from its printed representation ?

### This printed representation is null , but the __repr()__ is \\x00'

In [11]:
chr(0).__repr__()

"'\\x00'"

## 2.1 c) What happens when this character occurs in text? It may be helpful to play around with the following in your Python interpreter and see if it matches your expectations.

### chr(0) prints \\x00 , but if we wrap around in print() , it prints blank. Hence it is formatting of null character.

# 2.2 Unicode Encodings

## 2.2 a) What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various input strings.

Key reasons to prefer UTF-8:
1. Smaller base vocabulary
UTF-8: 256 possible byte values → base vocabulary of 256
UTF-16: 65,536 possible 2-byte units → much larger base vocabulary
UTF-32: 4 billion possible values → impractically large
A smaller base vocabulary means BPE can learn more meaningful merges rather than memorizing rare codepoints.

2. Space efficiency for common text
ASCII characters (most code, English text) are 1 byte in UTF-8, but 2 bytes in UTF-16 and 4 bytes in UTF-32
Training corpora are often ASCII-heavy, so UTF-8 is significantly more compact
3. No null bytes or BOM issues
UTF-16/32 have null bytes (0x00) throughout ASCII text, wasting space
UTF-16/32 require byte-order marks (BOM) adding overhead
UTF-8 has no nulls in the ASCII range and no BOM needed


## 2.2 b) Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string into a Unicode string. Why is this function incorrect? Provide an example of an input byte string that yields incorrect results.



In [15]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])
# >>> decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))
# 'hello'

In [16]:
decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

'hello'

### Example where it fails

In [17]:
decode_utf8_bytes_to_str_wrong("café".encode("utf-8"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc3 in position 0: unexpected end of data

### The function is incorrect because it attempts to decode each byte independently, but multi-byte UTF-8 characters (any non-ASCII character) require their constituent bytes to be decoded together as a unit.

In [19]:
### Corrected code

def decode_utf8_bytes_to_str_correct(bytestring: bytes):
    return bytestring.decode("utf-8")  # Decode the entire byte sequence at once


In [20]:
decode_utf8_bytes_to_str_correct("café".encode("utf-8"))

'café'

Other examples of invalid 2-byte sequences:
Sequence	Why it's invalid
b'\xff\xfe'	0xFF is never valid in UTF-8
b'\xc3\x28'	Leading byte 0xC3 expects continuation byte (0x80-0xBF), but 0x28 is ASCII (
b'\xc0\x80'	"Overlong encoding" — 0xC0-0xC1 are forbidden as they would encode ASCII characters inefficiently


>>> b'\x80\x80'.decode('utf-8')
# UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 0: invalid start byte
